In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import os, zipfile, shutil


file_name = "archive (6).zip"
drive_path = f"/content/drive/MyDrive/{file_name}"


extract_to = "/content/dataset"
os.makedirs(extract_to, exist_ok=True)

In [ ]:

if file_name.endswith(".zip"):
    with zipfile.ZipFile(drive_path, 'r') as z:
        z.extractall(extract_to)
        print("✅ Extracted ZIP successfully!")

elif file_name.endswith((".tar.gz", ".tgz")):
    import tarfile
    with tarfile.open(drive_path, 'r:gz') as t:
        t.extractall(extract_to)
        print("✅ Extracted TAR.GZ successfully!")

✅ Extracted ZIP successfully!


In [ ]:

for root, dirs, files in os.walk(extract_to):
    level = root.replace(extract_to, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:5]:
        print(f"  {indent}{f}")
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

dataset/
  hagrid-sample-30k-384p/
    hagrid_30k/
      train_val_like/
        6aebd85a-337a-4e73-975b-7388be3b0be5.jpg
        f6920f7d-e166-40ea-8433-f3f74a3c33ec.jpg
        a04be743-0194-48ee-a61b-57b7640ab798.jpg
        188ca28a-dc33-4779-82a4-084ac4701e69.jpg
        9e829a15-0579-47ea-8a9a-f792bf0f4157.jpg
      train_val_four/
        c2ba27cf-c1af-4b64-bb69-c17df02c121c.jpg
        73401211-4915-4c85-8dbc-e3552a7b9abd.jpg
        a5d956b7-1d5e-4f41-abee-8ffbdfa6d80f.jpg
        17c5c933-a315-43c5-a45a-fdbe5fe69d4a.jpg
        e02b0794-1a57-48f5-9b0b-2e41fcc72d37.jpg
      train_val_stop/
        ad40e808-44d7-4293-beb6-9761a7f8b17a.jpg
        bddc2ba9-b6f7-4ba7-9ebe-18cc52c5b1a0.jpg
        ab08206b-cdb3-4ada-bcc3-e3cd20dd2a61.jpg
        f3375fb4-1cbf-426d-93bb-a97daf17138f.jpg
        e40aa80b-d2c9-4bdd-a2da-54e450a0bfe6.jpg
      train_val_rock/
        cfdbeb0d-49c9-4d0b-b9f4-18c2832c8fe2.jpg
        282c6970-4b40-4bb7-820e-4da933b6a0f2.jpg
        0fb4c53b-4b07-4dfc-9

In [ ]:

import os, json, shutil, random
from pathlib import Path
from PIL import Image
import yaml
from tqdm import tqdm

SEED = 42
random.seed(SEED)

In [ ]:

DATA_ROOT = "/content/dataset/hagrid-sample-30k-384p/hagrid_30k"
ANN_ROOT  = "/content/dataset/hagrid-sample-30k-384p/ann_train_val"
YOLO_DS   = "/content/hagrid_yolo"
VAL_RATIO = 0.2 # 20% data used for validation
IMG_SIZE  = 640 #reszie
EPOCHS    = 30
BATCH     = 16 # 16images in 1 batch

In [ ]:

class_dirs = sorted([
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d)) and d.startswith("train_val_")
])
classes      = [d.replace("train_val_", "") for d in class_dirs]
class_to_idx = {c: i for i, c in enumerate(classes)} #finding all 18 gestures int he list
print(f"Classes ({len(classes)}): {classes}")

Classes (18): ['call', 'dislike', 'fist', 'four', 'like', 'mute', 'ok', 'one', 'palm', 'peace', 'peace_inverted', 'rock', 'stop', 'stop_inverted', 'three', 'three2', 'two_up', 'two_up_inverted']


In [ ]:

def build_yolo_dataset():
    for split in ["train", "val"]:
        (Path(YOLO_DS)/"images"/split).mkdir(parents=True, exist_ok=True)
        (Path(YOLO_DS)/"labels"/split).mkdir(parents=True, exist_ok=True)

    ann_files = sorted(Path(ANN_ROOT).glob("*.json")) #finds all annotation files
    print(f"Found {len(ann_files)} annotation files")

    for ann_file in tqdm(ann_files, desc="Converting"):
        gesture = ann_file.stem #getting filename widout extention
        if gesture not in class_to_idx:
            continue
        class_id = class_to_idx[gesture]
        img_dir  = os.path.join(DATA_ROOT, f"train_val_{gesture}")

        with open(ann_file) as f:
            annotations = json.load(f) #reads json file into python dict

        img_ids = list(annotations.keys())
        random.shuffle(img_ids)
        cut = int(len(img_ids) * (1 - VAL_RATIO))

        for i, img_id in enumerate(img_ids):
            split    = "train" if i < cut else "val"
            ann      = annotations[img_id]
            img_path = os.path.join(img_dir, f"{img_id}.jpg")
            if not os.path.exists(img_path):
                continue


            shutil.copy2(img_path, Path(YOLO_DS)/"images"/split/f"{img_id}.jpg")


            with open(Path(YOLO_DS)/"labels"/split/f"{img_id}.txt", "w") as lf:
                for bbox, lbl in zip(ann.get("bboxes", []), #looping thru bounding boxes and its label in the annotation
                                     ann.get("labels", [gesture]*len(ann.get("bboxes",[])))):
                    cid      = class_to_idx.get(lbl, class_id)#getting numeric class id for this specific label
                    x, y, w, h = bbox          # already normalized in HaGRID
                    xc, yc   = x + w/2, y + h/2 #center for yolo
                    lf.write(f"{cid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")#writing in txt file

build_yolo_dataset()
print(f"Train imgs: {len(list((Path(YOLO_DS)/'images'/'train').glob('*.jpg')))}")
print(f"Val imgs:   {len(list((Path(YOLO_DS)/'images'/'val').glob('*.jpg')))}")

Found 18 annotation files


Converting: 100%|██████████| 18/18 [00:16<00:00,  1.09it/s]


Train imgs: 25406
Val imgs:   6427


In [ ]:

yaml_path = "/content/hagrid.yaml"
with open(yaml_path, "w") as f:
    yaml.dump({
        "path"  : YOLO_DS,
        "train" : "images/train",
        "val"   : "images/val",
        "nc"    : len(classes),
        "names" : classes,
    }, f, default_flow_style=False) #human readability
print(open(yaml_path).read())

names:
- call
- dislike
- fist
- four
- like
- mute
- ok
- one
- palm
- peace
- peace_inverted
- rock
- stop
- stop_inverted
- three
- three2
- two_up
- two_up_inverted
nc: 18
path: /content/hagrid_yolo
train: images/train
val: images/val



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

DRIVE_SAVE_DIR = "/content/drive/MyDrive/hagrid_checkpoints"

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

print("Checkpoint folder:", DRIVE_SAVE_DIR)



Checkpoint folder: /content/drive/MyDrive/hagrid_checkpoints


In [ ]:
from ultralytics import YOLO
import shutil

model = YOLO("yolov8n.pt")

results = model.train(
    data          = yaml_path,
    epochs        = EPOCHS,
    imgsz         = IMG_SIZE,
    batch         = BATCH,
    lr0           = 1e-3, #initial learning rate
    lrf           = 0.01,
    warmup_epochs = 3,
    optimizer     = "AdamW",
    fliplr        = 0.5, #flip, augmentation , flips randomly
    degrees       = 15.0,#slight rotation
    project       = "/content/yolo_runs",
    name          = "hagrid",
    exist_ok      = True,
    device        = 0,
    patience      = 10, #if no improvment for 10 epochs stop early
    save          = True,
    plots         = True,


    save_period   = 1
)


src = "/content/yolo_runs/hagrid"
dst = os.path.join(DRIVE_SAVE_DIR, "hagrid_latest")

if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Training checkpoints backed up to Drive!")
print("Saved at:", dst)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hagrid.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, in

In [ ]:

!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.1 MB/s eta 0:00:00


In [ ]:
import os

save_dir = '/content/drive/MyDrive/asl_yolov8_training'

print("📂 Files in your training folder:")
for root, dirs, files in os.walk(save_dir):
    level = root.replace(save_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

📂 Files in your training folder:


In [ ]:
!pip install gtts -q

import cv2#opencv to decode webcam frames and draw on images
import numpy as np#converting raw bytes to image arrays
import time
import threading #for making voice run in bg so detection deosnt stop, two things at a time
import ipywidgets as widgets#for ui,the live image display
from IPython.display import display, Audio#to play audio, and run js in browser
from google.colab.output import eval_js#executes js and returns res to python
from base64 import b64decode#decodes the base64 image string from the webcam, images travel from browser b64decode converts text back into raw image bytes
from gtts import gTTS
from ultralytics import YOLO
from IPython.display import Javascript#used to open the webcam which lives in the browser not python


model = YOLO('/content/drive/MyDrive/hagrid_checkpoints/hagrid_latest/weights/best.pt')


_speaking = threading.Event()#flag

def speak_async(text):
    if _speaking.is_set(): #if true means voice is curr playing
        return
    def _speak():
        _speaking.set()
        try: #so nothing crashes
            tts = gTTS(text=text, lang='en')
            tts.save("voice.mp3")
            display(Audio("voice.mp3", autoplay=True))#saving as mp3 audio file
            time.sleep(len(text) * 0.065 + 0.8)#controls how fast detection runs
        finally:
            _speaking.clear()
    threading.Thread(target=_speak, daemon=True).start() #means if the main program stops, thread also stops auto


display(Javascript('''
    window._camReady = false;
    window._camStream = null;  //stores webcam stream obj
    (async () => { //use await for things like camera permission
        const video = document.createElement('video'); //to display webcam feed
        video.id = 'colab_cam';
        video.style.display = 'none';
        video.autoplay = true;
        video.playsinline = true;
        document.body.appendChild(video);//ads hidden vid element to webpage body
        const stream = await navigator.mediaDevices.getUserMedia({video: true});//requesting webcam access,video = true means only vid
        window._camStream = stream;
        video.srcObject = stream;
        await video.play();
        await new Promise(r => setTimeout(r, 1500));
        window._camReady = true;
    })();
''')) #everything inside ' ' ' is js

print("Waiting for camera to initialise (2 sec)...")
time.sleep(2.5)


title_html = widgets.HTML(
    "<h2 style='font-family:monospace;color:#00ff99;background:#111;"
    "padding:10px 20px;margin:0;letter-spacing:2px;'>🤟 LIVE HAGRID DETECTOR</h2>"
)
img_widget = widgets.Image(format='jpeg', width=640)
label_box  = widgets.HTML(
    "<p style='font-family:monospace;font-size:13px;background:#111;"
    "color:#eee;padding:8px 16px;margin:0;'>Starting…</p>"
)
stop_btn = widgets.Button(
    description="⏹ Stop", button_style='danger',
    layout=widgets.Layout(width='120px', margin='6px')
)
panel = widgets.VBox(
    [title_html, img_widget, label_box, stop_btn],
    layout=widgets.Layout(border='2px solid #00ff99', width='fit-content')
)
display(panel)


running = [True]
def on_stop(_):
    running[0] = False
    label_box.value = ("<p style='font-family:monospace;background:#111;"
                       "color:#ff4444;padding:8px 16px;margin:0;'>⏹ Stopped.</p>")
stop_btn.on_click(on_stop)


GRAB_JS = '''
async function grabFrame() {
    const video = document.getElementById('colab_cam');
    if (!video || !window._camReady) return null;
    const canvas = document.createElement('canvas');
    canvas.width  = video.videoWidth  || 640;
    canvas.height = video.videoHeight || 480;
    canvas.getContext('2d').drawImage(video, 0, 0);
    return canvas.toDataURL('image/jpeg', 0.85);
}
grabFrame()
'''

def grab_frame():
    data = eval_js(GRAB_JS)
    if not data or data == 'null':
        return None
    img_bytes = b64decode(data.split(',')[1])
    arr = np.frombuffer(img_bytes, np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)


gesture_speech = {
    'call':           'Call gesture detected',
    'dislike':        'Thumbs down',
    'fist':           'Fist detected',
    'four':           'Four fingers',
    'like':           'Thumbs up',
    'mute':           'Mute gesture',
    'ok':             'OK sign',
    'one':            'One finger',
    'palm':           'Palm detected',
    'peace':          'Peace sign',
    'peace_inverted': 'Inverted peace sign',
    'rock':           'Rock on',
    'stop':           'Stop gesture',
    'stop_inverted':  'Inverted stop',
    'three':          'Three fingers',
    'three2':         'Three fingers variant',
    'two_up':         'Two fingers up',
    'two_up_inverted':'Two fingers inverted',
}


print("Stream running — click ⏹ Stop to end.\n")
last_spoken = ""
frame_count = 0

while running[0]:
    frame = grab_frame()
    if frame is None:
        label_box.value = ("<p style='font-family:monospace;background:#111;"
                           "color:orange;padding:8px;'>⚠ Waiting for camera…</p>")
        time.sleep(0.3)
        continue

    frame_count += 1

    results   = model.predict(source=frame, conf=0.40, verbose=False)
    result    = results[0]
    annotated = result.plot()

    _, buf = cv2.imencode('.jpg', annotated, [cv2.IMWRITE_JPEG_QUALITY, 85])
    img_widget.value = buf.tobytes()

    boxes = result.boxes
    if len(boxes) == 0:
        speech     = "No gesture detected."
        label_html = "<span style='color:#666;'>No detections</span>"
    else:
        detected = {}
        for box in boxes:
            cls_name = model.names[int(box.cls)]
            conf     = float(box.conf)
            if cls_name not in detected or conf > detected[cls_name]:
                detected[cls_name] = conf

        detections = sorted(detected.items(), key=lambda x: x[1], reverse=True)
        label_html = "".join(
            f"<span style='background:#00ff9922;border:1px solid #00ff99;"
            f"border-radius:4px;padding:2px 10px;margin-right:6px;color:#00ff99;"
            f"font-weight:bold;'>{c} {v:.0%}</span>"
            for c, v in detections[:5]
        )

        top = detections[0][0]
        speech = gesture_speech.get(top, f"I see {top}")

    mic = " 🔊" if _speaking.is_set() else ""
    label_box.value = (
        f"<div style='font-family:monospace;font-size:13px;background:#111;"
        f"color:#eee;padding:8px 16px;margin:0;'>"
        f"<span style='color:#555;margin-right:12px;'>#{frame_count}{mic}</span>"
        f"{label_html}</div>"
    )

    if speech != last_spoken and not _speaking.is_set():
        speak_async(speech)
        last_spoken = speech

    time.sleep(0.1)


display(Javascript("if(window._camStream) window._camStream.getTracks().forEach(t=>t.stop());"))
print("Camera released.")